# 01 — Panel EDA
CMS-calibrated 800-patient Medicare Advantage panel with wearable coverage analysis.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("pcp-panel-intelligence"):
        os.system("git clone https://github.com/Hannah-Hiltz/pcp-panel-intelligence.git")
    os.chdir("pcp-panel-intelligence")
    sys.path.insert(0, "src")
    os.system("pip install -q xgboost shap scikit-learn joblib")
    DATA_DIR  = "data/raw";      PROC_DIR  = "data/processed"
    FIG_DIR   = "reports/figures"; MODEL_DIR = "models"
    WEAR_DIR  = "data/raw/wearables"; PANEL_DIR = "data/raw/panel"
else:
    sys.path.insert(0, "../src")
    DATA_DIR  = "../data/raw";   PROC_DIR  = "../data/processed"
    FIG_DIR   = "../reports/figures"; MODEL_DIR = "../models"
    WEAR_DIR  = "../data/raw/wearables"; PANEL_DIR = "../data/raw/panel"

for d in [FIG_DIR, PROC_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print(f"Environment: {'Colab' if IN_COLAB else 'Jupyter'}")


In [ ]:
df = pd.read_csv(f"{PANEL_DIR}/patient_panel_flat.csv")
print(f"Panel: {len(df):,} patients | Wearable coverage: {df['has_wearable'].mean()*100:.1f}%")
df.head()


## Priority distribution

In [ ]:
order  = ["Critical","High","Moderate","Routine"]
colors = {"Critical":"#E24B4A","High":"#EF9F27","Moderate":"#97C459","Routine":"#5DCAA5"}
counts = df["outreach_priority"].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(counts.index, counts.values, color=[colors[p] for p in counts.index])
ax.set_title("Priority distribution — CMS-calibrated 800-patient panel", fontsize=13)
ax.set_ylabel("Patients")
for i,v in enumerate(counts.values):
    ax.text(i, v+3, f"{v} ({v/len(df)*100:.0f}%)", ha="center", fontsize=10)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/priority_dist.png", dpi=150, bbox_inches="tight"); plt.show()


## CMS prevalence calibration check

In [ ]:
cms = pd.read_csv(f"{DATA_DIR}/cms_puf/cms_chronic_conditions_puf.csv")
print("CMS PUF — national MA prevalence rates:")
print(cms[["age_category","gender","dual_eligible","diabetes","hypertension","hyperlipidemia"]].to_string(index=False))
cond_cols = [c for c in df.columns if c.startswith("has_")]
print("\nSynthetic panel prevalence:")
for c in cond_cols:
    print(f"  {c.replace('has_',''):25s} {df[c].mean()*100:.1f}%")


## Wearable ownership by income quintile — equity analysis

In [ ]:
equity = df.groupby("zip_income_quintile").agg(
    n=("patient_id","count"),
    pct_device=("has_wearable","mean"),
    avg_completeness=("wear_completeness","mean"),
).round(3).reset_index()
equity["pct_device"] *= 100
equity["avg_completeness"] *= 100

fig, axes = plt.subplots(1,2, figsize=(13,5))
axes[0].bar(equity["zip_income_quintile"], equity["pct_device"], color="#378ADD")
axes[0].set_title("Wearable ownership by income quintile", fontsize=13)
axes[0].set_xlabel("Income quintile (1=lowest)"); axes[0].set_ylabel("% with device")
axes[0].axhline(df["has_wearable"].mean()*100, linestyle="--", color="grey", label="Panel avg")
axes[0].legend()

axes[1].bar(equity["zip_income_quintile"], equity["avg_completeness"], color="#5DCAA5")
axes[1].set_title("Avg data completeness by income quintile", fontsize=13)
axes[1].set_xlabel("Income quintile (1=lowest)"); axes[1].set_ylabel("% days with data")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/wearable_equity.png", dpi=150, bbox_inches="tight"); plt.show()
print("Key finding: Q1 (lowest income) owns devices 30%+ less than Q5.")
print("Conservative-upward imputation strategy preserves equity in risk scoring.")


## SDOH distribution

In [ ]:
sdoh_cols  = ["food_insecurity","housing_instability","transportation_barrier"]
sdoh_labels = ["Food insecurity","Housing instability","Transport barrier"]
fig, axes = plt.subplots(1,3, figsize=(14,5))
colors_list = ["#E24B4A","#EF9F27","#97C459","#5DCAA5"]
for ax,col,label in zip(axes, sdoh_cols, sdoh_labels):
    (df.groupby("outreach_priority")[col].mean()*100).reindex(order).plot(kind="bar", ax=ax, color=colors_list)
    ax.set_title(f"{label} by priority", fontsize=11)
    ax.set_ylabel("% patients"); ax.set_xlabel(""); ax.tick_params(axis="x",rotation=30)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sdoh_dist.png", dpi=150, bbox_inches="tight"); plt.show()
